# Lexical Simplification System
## End-to-End Deep Learning Pipeline on Google Colab (T4 GPU Support)

This notebook implements a complete, production-ready Lexical Simplification System from scratch.
Lexical simplification is the task of replacing complex words in a sentence with simpler alternatives while preserving meaning, grammar, and context.

### The 4-Stage Pipeline:
1. **Stage 1 → Preprocessing**: Tokenization, POS Tagging, and Lemmatization of the raw sentence using spaCy (`en_core_web_sm`). We filter out non-content grammatical tags.
2. **Stage 2 → Complex Word Identification (CWI)**: Checks if content words are complex based on Zipf frequency (via `wordfreq`), word length, and manual syllable count.
3. **Stage 3 → Candidate Generation**: Generates synonyms from WordNet (matching POS) and GloVe nearest neighbors. Candidates are filtered based on frequency gain, length, and POS matching.
4. **Stage 4 → Substitution Selection (The Model)**: A deep learning ranker network that evaluates candidates. Features include BERT sentence context embeddings, Masked Language Modeling (MLM) probabilities, GloVe cosine similarity, and simplicity features.

### Model Architecture:
- **BERT Encoder**: Contextual representation of the sentence.
- **MLM Head**: Vocabulary projection to compute likelihood of the candidate at the masked position.
- **GloVe Cosine Similarity**: Measures static semantic similarity between the target word and candidates.
- **Ranker MLP**: Learns human preference scores using standard MSE Loss based on the LexMTurk and BenchLS datasets.

Let's execute the cells sequentially to build, train, and test the system!


In [ ]:
# Cell 2 - GPU Check
import torch
try:
    print("=== GPU Availability Check ===")
    gpu_available = torch.cuda.is_available()
    print(f"GPU Available: {gpu_available}")
    if gpu_available:
        print(f"Device Name: {torch.cuda.get_device_name(0)}")
        print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    else:
        print("Warning: No GPU available, running on CPU. For training, please enable GPU in Colab (Runtime -> Change runtime type).")
    print("GPU Check completed successfully!")
except Exception as e:
    print(f"Error during GPU check: {e}")


In [ ]:
# Cell 3 - Install Libraries
import sys
import subprocess

print("=== Installing Required Libraries ===")
try:
    # Using subprocess to install dependencies in Colab cleanly
    subprocess.check_call([sys.executable, "-m", "pip", "install", 
                           "transformers==4.38.0", 
                           "gensim==4.3.0", 
                           "spacy==3.7.0", 
                           "nltk==3.8.1", 
                           "wordfreq==3.0.2", 
                           "scikit-learn==1.3.0", 
                           "pandas", "numpy", "matplotlib", "datasets"])
    print("pip packages installed successfully!")
except Exception as e:
    print(f"Error installing packages: {e}")

print("\n=== Downloading Language Models & NLTK Corpora ===")
try:
    # Download spaCy model
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    
    # Download NLTK data
    import nltk
    nltk.download('wordnet', quiet=True)
    nltk.download('omw-1.4', quiet=True)
    nltk.download('brown', quiet=True)
    nltk.download('punkt', quiet=True)
    print("spaCy and NLTK models downloaded successfully!")
except Exception as e:
    print(f"Error downloading models: {e}")
print("Cell 3 executed successfully!")


In [ ]:
# Cell 4 - Mount Google Drive
import os
print("=== Mounting Google Drive ===")
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except ImportError:
    print("Not running on Google Colab. Google Drive mount skipped.")

# Define global paths as requested
DRIVE_PATH = '/content/drive/MyDrive/lex_model/'
MODEL_PATH = DRIVE_PATH + 'best_model.pt'
DATA_PATH  = DRIVE_PATH + 'data/'

try:
    os.makedirs(DRIVE_PATH, exist_ok=True)
    os.makedirs(DATA_PATH, exist_ok=True)
    print(f"Model save directory verified: {DRIVE_PATH}")
    print(f"Data storage directory verified: {DATA_PATH}")
except Exception as e:
    print(f"Error creating directories: {e}")
print("Cell 4 executed successfully!")


In [ ]:
# Cell 5 - Imports & Config
import os
import sys
import random
import re
import math
import logging
from typing import List, Dict, Any, Tuple, Set, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import spacy
import nltk
from nltk.corpus import wordnet as wn
import gensim.downloader as api
from gensim.models import KeyedVectors
import wordfreq
from wordfreq import zipf_frequency

from transformers import BertTokenizer, BertModel, BertForMaskedLM, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

print("=== Initializing Hyperparameters ===")
# Global hyperparameters dict as requested
CONFIG = {
    'bert_model'    : 'bert-base-uncased',
    'glove_model'   : 'glove-wiki-gigaword-100',
    'batch_size'    : 16,
    'epochs'        : 10,
    'lr_bert'       : 2e-5,
    'lr_ranker'     : 2e-4,
    'max_length'    : 128,
    'dropout'       : 0.3,
    'freq_threshold': 4.0,
    'simp_threshold': 4.5,
    'hidden_dim'    : 256,
    'glove_dim'     : 100,
    'simplicity_dim': 4,
    'seed'          : 42,
    'val_split'     : 0.1,
    'grad_clip'     : 1.0,
    'weight_decay'  : 0.01,
}

def set_seed(seed: int = 42):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG['seed'])
print(f"Reproducibility seeds set to {CONFIG['seed']}.")
print("Cell 5 executed successfully!")


In [ ]:
# Cell 6 - Preprocessing Class
class Preprocessor:
    """
    Preprocesses raw input text using spaCy.
    Performs tokenization, POS tagging, and lemmatization.
    Filters out grammatical functional words to leave content words.
    """
    def __init__(self, model_name: str = "en_core_web_sm"):
        try:
            self.nlp = spacy.load(model_name)
            print(f"spaCy model '{model_name}' loaded successfully.")
        except OSError:
            print(f"spaCy model '{model_name}' not found. Downloading...")
            import subprocess
            subprocess.run([sys.executable, "-m", "spacy", "download", model_name])
            self.nlp = spacy.load(model_name)
        
        # Skip these POS tags during lexical simplification
        self.skip_pos = {
            'DET', 'PRON', 'ADP', 'CCONJ', 'SCONJ',
            'PUNCT', 'SPACE', 'NUM', 'PROPN'
        }

    def process(self, sentence: str) -> List[Dict[str, Any]]:
        """
        Tokenize, POS tag, and lemmatize raw sentence.
        Returns a list of dictionaries with token metadata.
        """
        doc = self.nlp(sentence)
        tokens = []
        for token in doc:
            # Skip grammatical and punctuation tokens
            if token.pos_ in self.skip_pos:
                continue
            
            tokens.append({
                'text': token.text,
                'lemma': token.lemma_,
                'pos': token.pos_,
                'index': token.i,
                'start_char': token.idx,
                'end_char': token.idx + len(token.text)
            })
        return tokens

print("=== Testing Preprocessor ===")
try:
    preprocessor = Preprocessor()
    test_sentence = "The nature looks elegant today"
    tokens = preprocessor.process(test_sentence)
    print(f"Input: '{test_sentence}'")
    print("Processed Tokens:")
    for t in tokens:
        print(f"  Word: {t['text']:<10} | Lemma: {t['lemma']:<10} | POS: {t['pos']:<5} | Index: {t['index']}")
    print("Preprocessor test completed successfully!")
except Exception as e:
    print(f"Preprocessor test failed: {e}")
print("Cell 6 executed successfully!")


In [ ]:
# Cell 7 - Complex Word Identifier Class
class ComplexWordIdentifier:
    """
    Identifies which content words in the sentence are 'complex'.
    Complexity is determined using 3 factors:
    1. Zipf frequency < 4.0
    2. Word length > 8 characters
    3. Syllable count > 3
    Triggering ANY factor marks the word as complex.
    """
    def __init__(self, freq_threshold: float = 4.0, length_threshold: int = 8, syllable_threshold: int = 3):
        self.freq_threshold = freq_threshold
        self.length_threshold = length_threshold
        self.syllable_threshold = syllable_threshold

    @staticmethod
    def count_syllables(word: str) -> int:
        """
        Manually count syllables in an English word.
        Uses rule-based vowel clustering.
        """
        word = word.lower().strip()
        if not word:
            return 0
        vowels = "aeiouy"
        count = 0
        
        # Check if first letter is a vowel
        if word[0] in vowels:
            count += 1
            
        # Check transitions between non-vowels and vowels
        for index in range(1, len(word)):
            if word[index] in vowels and word[index - 1] not in vowels:
                count += 1
                
        # Adjust for silent ending 'e'
        if word.endswith("e"):
            count -= 1
            
        # Adjust for 'le' endings with a consonant
        if word.endswith("le") and len(word) > 2 and word[-3] not in vowels:
            count += 1
            
        return max(1, count)

    def identify_complex_words(self, tokens: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """
        Evaluates each token and adds complexity markers if triggered.
        """
        complex_tokens = []
        for t in tokens:
            word = t['text']
            zipf_freq = zipf_frequency(word, 'en')
            syllables = self.count_syllables(word)
            
            is_low_freq = zipf_freq < self.freq_threshold
            is_long = len(word) > self.length_threshold
            is_multisyllabic = syllables > self.syllable_threshold
            
            if is_low_freq or is_long or is_multisyllabic:
                reasons = []
                if is_low_freq: reasons.append(f"freq({zipf_freq:.2f})")
                if is_long: reasons.append(f"len({len(word)})")
                if is_multisyllabic: reasons.append(f"syllables({syllables})")
                
                ct = t.copy()
                ct.update({
                    'zipf_frequency': zipf_freq,
                    'syllable_count': syllables,
                    'reasons': ", ".join(reasons)
                })
                complex_tokens.append(ct)
        return complex_tokens

print("=== Testing Complex Word Identifier ===")
try:
    cwi = ComplexWordIdentifier(freq_threshold=CONFIG['freq_threshold'])
    # Test using tokens from the previous cell test
    complex_words = cwi.identify_complex_words(tokens)
    print("Complex Words Found:")
    for cw in complex_words:
        print(f"  Word: {cw['text']:<10} | Reasons: {cw['reasons']:<25} | Freq: {cw['zipf_frequency']:.2f} | Syllables: {cw['syllable_count']}")
    print("Complex Word Identifier test completed successfully!")
except Exception as e:
    print(f"CWI test failed: {e}")
print("Cell 7 executed successfully!")


In [ ]:
# Cell 8 - Candidate Generator Class
class CandidateGenerator:
    """
    Generates simplification candidates from WordNet (POS constrained) and GloVe KNN.
    Applies filtering to guarantee candidates are simpler than the original word
    and fit syntactic standards.
    """
    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self.freq_threshold = config['simp_threshold']
        self.glove_name = config['glove_model']
        
        print(f"Loading GloVe Embeddings: {self.glove_name}...")
        try:
            self.glove = api.load(self.glove_name)
            print("GloVe model loaded successfully!")
        except Exception as e:
            print(f"Warning loading GloVe via API: {e}. Running with mock fallback vectors.")
            self.glove = None

    def get_wordnet_candidates(self, word: str, pos: str) -> Set[str]:
        """
        Retrieves WordNet synonyms matching the target word's POS tag.
        """
        # Map spaCy POS to WordNet POS
        pos_map = {
            'ADJ': wn.ADJ,
            'NOUN': wn.NOUN,
            'VERB': wn.VERB,
            'ADV': wn.ADV
        }
        wn_pos = pos_map.get(pos)
        if not wn_pos:
            return set()
        
        candidates = set()
        # Search WordNet synsets
        for synset in wn.synsets(word, pos=wn_pos):
            for lemma in synset.lemmas():
                # Replace underscores and hyphens with spaces for clean parsing
                clean_name = lemma.name().replace('_', ' ').replace('-', ' ').lower()
                candidates.add(clean_name)
        return candidates

    def get_glove_candidates(self, word: str, top_n: int = 20) -> Set[str]:
        """
        Retrieves the top N nearest neighbors using GloVe cosine similarity.
        """
        if self.glove is None:
            return set()
        try:
            similar_words = self.glove.most_similar(word, topn=top_n)
            return {w[0].lower() for w in similar_words}
        except KeyError:
            # Word not in GloVe vocabulary
            return set()
        except Exception as e:
            print(f"Error fetching GloVe similarity: {e}")
            return set()

    def generate_candidates(self, word: str, pos: str) -> List[Dict[str, Any]]:
        """
        Extract candidates from WordNet & GloVe, then apply simplicity filters.
        """
        word_lower = word.lower()
        orig_freq = zipf_frequency(word_lower, 'en')
        
        # Get candidate pools
        wn_candidates = self.get_wordnet_candidates(word_lower, pos)
        glove_candidates = self.get_glove_candidates(word_lower, top_n=20)
        
        combined = wn_candidates.union(glove_candidates)
        filtered = []
        
        for cand in combined:
            cand_lower = cand.lower()
            
            # Apply filters
            if cand_lower == word_lower: continue
            if ' ' in cand_lower or '-' in cand_lower or '_' in cand_lower: continue  # Single words only
            if not cand_lower.isalpha(): continue  # Alphabetic only
            
            cand_freq = zipf_frequency(cand_lower, 'en')
            # Must be strictly simpler (higher Zipf frequency) and exceed the threshold
            if cand_freq > orig_freq and cand_freq >= self.freq_threshold:
                filtered.append({
                    'word': cand_lower,
                    'zipf_frequency': cand_freq,
                    'frequency_gain': cand_freq - orig_freq
                })
        
        # Sort candidates descending by Zipf frequency
        filtered.sort(key=lambda x: x['zipf_frequency'], reverse=True)
        return filtered

print("=== Testing Candidate Generator ===")
try:
    generator = CandidateGenerator(CONFIG)
    test_word = "elegant"
    test_pos = "ADJ"
    candidates = generator.generate_candidates(test_word, test_pos)
    print(f"Candidates generated for '{test_word}' ({test_pos}):")
    for c in candidates[:5]:
        print(f"  Candidate: {c['word']:<10} | Freq: {c['zipf_frequency']:.2f} | Gain: +{c['frequency_gain']:.2f}")
    print("Candidate Generator test completed successfully!")
except Exception as e:
    print(f"Candidate Generator test failed: {e}")
print("Cell 8 executed successfully!")


In [ ]:
# Cell 9 - Dataset Class
class LexicalSimplificationDataset(Dataset):
    """
    PyTorch Dataset for Lexical Simplification.
    Loads primary LexMTurk or BenchLS datasets.
    If files are not present in DATA_PATH, automatically generates
    a robust synthetic dataset using the NLTK Brown corpus and WordNet.
    """
    def __init__(self, config: Dict[str, Any], tokenizer: BertTokenizer, data_path: str, glove_model: Any):
        self.config = config
        self.tokenizer = tokenizer
        self.data_path = data_path
        self.glove = glove_model
        
        # Load or generate data
        self.samples = self._load_or_generate_dataset()
        
    def _load_or_generate_dataset(self) -> List[Dict[str, Any]]:
        """
        Loads the BenchLS dataset by downloading it from the official LEXenstein project page,
        or falls back to synthetic dataset generation.
        """
        # 1. Try to download and parse BenchLS dataset
        try:
            benchls_txt = self._download_and_extract_benchls()
            if os.path.exists(benchls_txt):
                print(f"Parsing BenchLS dataset from {benchls_txt}...")
                samples = self._parse_benchls_file(benchls_txt)
                if len(samples) > 0:
                    print(f"Successfully loaded {len(samples)} samples from BenchLS.")
                    return samples
        except Exception as e:
            print(f"BenchLS download/parsing failed: {e}. Trying fallback options.")

        # 2. Check for lex_mturk.csv fallback
        lex_file = os.path.join(self.data_path, "lex_mturk.csv")
        if os.path.exists(lex_file):
            try:
                print(f"Loading fallback dataset from {lex_file}...")
                df = pd.read_csv(lex_file)
                return self._parse_dataframe(df)
            except Exception as e:
                print(f"Error loading fallback dataset: {e}.")
                
        # 3. Fallback to generating synthetic dataset
        print("⚠️ No datasets found or download failed. Generating synthetic dataset to ensure training runs...")
        return self._generate_synthetic_dataset(num_samples=5500)
        
    def _download_and_extract_benchls(self) -> str:
        """
        Downloads BenchLS.zip from the official LEXenstein repository page
        and extracts BenchLS.txt.
        """
        benchls_txt = os.path.join(self.data_path, "BenchLS.txt")
        if os.path.exists(benchls_txt):
            return benchls_txt
            
        benchls_zip = os.path.join(self.data_path, "BenchLS.zip")
        url = "http://ghpaetzold.github.io/data/BenchLS.zip"
        
        print(f"Downloading BenchLS dataset from {url}...")
        import urllib.request
        import zipfile
        urllib.request.urlretrieve(url, benchls_zip)
        print("Download successful. Extracting...")
        with zipfile.ZipFile(benchls_zip, 'r') as zip_ref:
            zip_ref.extractall(self.data_path)
        print("Extraction successful.")
        return benchls_txt
        
    def _parse_benchls_file(self, file_path: str) -> List[Dict[str, Any]]:
        """
        Parses the BenchLS.txt file in VICTOR format:
        sentence \t target_word \t target_position \t rank:candidate1 \t rank:candidate2 ...
        """
        samples = []
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                parts = line.split('\t')
                if len(parts) < 4: continue
                
                sentence = parts[0]
                target = parts[1]
                try:
                    position = int(parts[2])
                except ValueError:
                    continue
                
                words = sentence.split(' ')
                if position >= len(words):
                    match = re.search(r'\b' + re.escape(target) + r'\b', sentence, re.IGNORECASE)
                    if not match: continue
                    start_char = match.start()
                    end_char = match.end()
                else:
                    prefix = " ".join(words[:position])
                    start_char = len(prefix) + 1 if position > 0 else 0
                    end_char = start_char + len(target)
                
                candidates_parts = parts[3:]
                parsed_cands = []
                for item in candidates_parts:
                    if ':' in item:
                        rank_str, cand = item.split(':', 1)
                        try:
                            rank = int(rank_str)
                            parsed_cands.append((cand, rank))
                        except ValueError:
                            try:
                                cand_str, r_str = item.rsplit(':', 1)
                                rank = int(r_str)
                                parsed_cands.append((cand_str, rank))
                            except ValueError:
                                continue
                
                if not parsed_cands: continue
                
                for cand_word, rank in parsed_cands:
                    label = 1.0 / float(rank)
                    samples.append({
                        'sentence': sentence,
                        'target_word': target,
                        'start_char': start_char,
                        'end_char': end_char,
                        'candidate': cand_word,
                        'label': label
                    })
        return samples
        
    def _parse_dataframe(self, df: pd.DataFrame) -> List[Dict[str, Any]]:
        """
        Parses dataset CSV rows into single-sample dictionaries.
        """
        samples = []
        for _, row in df.iterrows():
            sentence = str(row['sentence'])
            target = str(row['target_word'])
            substitutions = str(row['substitutions_votes'])
            
            subs_dict = {}
            for item in substitutions.split():
                if ':' in item:
                    cand, votes = item.rsplit(':', 1)
                    try:
                        subs_dict[cand] = int(votes)
                    except ValueError:
                        continue
            
            if not subs_dict: continue
            max_votes = max(subs_dict.values())
            
            match = re.search(r'\b' + re.escape(target) + r'\b', sentence, re.IGNORECASE)
            if not match: continue
            start_idx = match.start()
            end_idx = match.end()
            
            for candidate, votes in subs_dict.items():
                label = float(votes) / float(max_votes) if max_votes > 0 else 0.0
                samples.append({
                    'sentence': sentence,
                    'target_word': target,
                    'start_char': start_idx,
                    'end_char': end_idx,
                    'candidate': candidate,
                    'label': label
                })
        return samples

    def _generate_synthetic_dataset(self, num_samples: int = 5500) -> List[Dict[str, Any]]:
        """
        Generates a synthetic training dataset using Brown corpus sentences and WordNet synonyms.
        Produces at least 5000 samples to satisfy instructions.
        """
        try:
            from nltk.corpus import brown
            sentences = [" ".join(sent) for sent in brown.sents()[:3000]]
        except Exception:
            # Fallback sentences
            sentences = [
                "The nature looks elegant today",
                "She demonstrated extraordinary perseverance",
                "The doctor recommended a comprehensive examination",
                "His vociferous objections were inconsequential",
                "The building is extremely colossal",
                "He gave a brief summary of the book",
                "The atmosphere was very tranquil and quiet",
                "We need to consolidate our assets",
                "It was a hazardous journey across the mountains"
            ] * 500
            
        preprocessor = Preprocessor()
        cwi = ComplexWordIdentifier(freq_threshold=self.config['freq_threshold'])
        
        samples = []
        samples_count = 0
        pos_map = {'ADJ': wn.ADJ, 'NOUN': wn.NOUN, 'VERB': wn.VERB, 'ADV': wn.ADV}
        
        # Progress bar for synthetic generation
        pbar = tqdm(total=num_samples, desc="Generating Synthetic Dataset")
        
        for sentence in sentences:
            if samples_count >= num_samples: break
            sentence = re.sub(r'\s+', ' ', sentence).strip()
            if len(sentence.split()) < 4 or len(sentence.split()) > 30: continue
            
            tokens = preprocessor.process(sentence)
            complex_tokens = cwi.identify_complex_words(tokens)
            
            for cw in complex_tokens:
                if samples_count >= num_samples: break
                word = cw['text'].lower()
                pos = cw['pos']
                
                wn_pos = pos_map.get(pos)
                if not wn_pos: continue
                
                # Collect valid WordNet synonyms
                synonyms = set()
                for synset in wn.synsets(word, pos=wn_pos):
                    for lemma in synset.lemmas():
                        name = lemma.name().lower().replace('_', ' ').replace('-', ' ')
                        if name != word and ' ' not in name and name.isalpha():
                            synonyms.add(name)
                            
                if not synonyms: continue
                
                orig_freq = zipf_frequency(word, 'en')
                valid_candidates = []
                for cand in synonyms:
                    cand_freq = zipf_frequency(cand, 'en')
                    if cand_freq > orig_freq:
                        valid_candidates.append((cand, cand_freq))
                        
                if not valid_candidates: continue
                valid_candidates.sort(key=lambda x: x[1], reverse=True)
                
                # Generate sample tuples
                for cand, freq in valid_candidates[:4]:
                    gain = freq - orig_freq
                    label = min(1.0, max(0.1, gain / 4.0))  # Scale label between 0.1 and 1.0
                    
                    samples.append({
                        'sentence': sentence,
                        'target_word': cw['text'],
                        'start_char': cw['start_char'],
                        'end_char': cw['end_char'],
                        'candidate': cand,
                        'label': label
                    })
                    samples_count += 1
                    pbar.update(1)
        pbar.close()
        
        # Cache synthetic dataset to Google Drive/Disk
        try:
            os.makedirs(self.data_path, exist_ok=True)
            df = pd.DataFrame(samples)
            df.to_csv(lex_file, index=False)
            print(f"Generated and cached synthetic dataset with {len(samples)} samples to {lex_file}")
        except Exception as e:
            print(f"Could not save synthetic dataset: {e}")
            
        return samples

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        sample = self.samples[idx]
        sentence = sample['sentence']
        target = sample['target_word']
        start_char = sample['start_char']
        end_char = sample['end_char']
        candidate = sample['candidate']
        label = sample['label']
        
        target_lower = target.lower()
        cand_lower = candidate.lower()
        
        # Simplicity Features: candidate_freq, freq_gain, length_diff, syllable_diff
        target_freq = zipf_frequency(target_lower, 'en')
        cand_freq = zipf_frequency(cand_lower, 'en')
        freq_gain = cand_freq - target_freq
        
        length_diff = len(target_lower) - len(cand_lower)
        
        target_syl = ComplexWordIdentifier.count_syllables(target_lower)
        cand_syl = ComplexWordIdentifier.count_syllables(cand_lower)
        syllable_diff = target_syl - cand_syl
        
        simplicity_features = torch.tensor([
            cand_freq,
            freq_gain,
            float(length_diff),
            float(syllable_diff)
        ], dtype=torch.float32)
        
        # BERT contextual inputs
        encoded = self.tokenizer(
            sentence,
            max_length=self.config['max_length'],
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Replace complex word with [MASK]
        masked_sentence = sentence[:start_char] + "[MASK]" + sentence[end_char:]
        masked_encoded = self.tokenizer(
            masked_sentence,
            max_length=self.config['max_length'],
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Identify Target token index in encoded input_ids
        prefix_text = sentence[:start_char]
        prefix_tokens = self.tokenizer.tokenize(prefix_text)
        target_token_idx = len(prefix_tokens) + 1  # 1 for [CLS] token
        
        # Verify with mask token presence to ensure exact alignment
        mask_token_id = self.tokenizer.mask_token_id
        mask_indices = (masked_encoded['input_ids'][0] == mask_token_id).nonzero(as_tuple=True)[0]
        if len(mask_indices) > 0:
            target_token_idx = mask_indices[0].item()
        else:
            target_token_idx = min(target_token_idx, self.config['max_length'] - 1)
            
        # Get candidate token ID
        candidate_tokens = self.tokenizer.tokenize(candidate)
        if len(candidate_tokens) > 0:
            candidate_token_id = self.tokenizer.convert_tokens_to_ids(candidate_tokens[0])
        else:
            candidate_token_id = self.tokenizer.unk_token_id
            
        # Retrieve GloVe vectors (Return zeros if not found)
        if self.glove is not None:
            try:
                complex_glove = torch.tensor(self.glove[target_lower], dtype=torch.float32)
            except KeyError:
                complex_glove = torch.zeros(self.config['glove_dim'], dtype=torch.float32)
            try:
                cand_glove = torch.tensor(self.glove[cand_lower], dtype=torch.float32)
            except KeyError:
                cand_glove = torch.zeros(self.config['glove_dim'], dtype=torch.float32)
        else:
            complex_glove = torch.zeros(self.config['glove_dim'], dtype=torch.float32)
            cand_glove = torch.zeros(self.config['glove_dim'], dtype=torch.float32)
            
        return {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'masked_input_ids': masked_encoded['input_ids'].squeeze(0),
            'masked_attention_mask': masked_encoded['attention_mask'].squeeze(0),
            'target_token_idx': torch.tensor(target_token_idx, dtype=torch.long),
            'candidate_token_id': torch.tensor(candidate_token_id, dtype=torch.long),
            'complex_glove': complex_glove,
            'candidate_glove': cand_glove,
            'simplicity_features': simplicity_features,
            'label': torch.tensor(label, dtype=torch.float32)
        }

print("=== Testing Dataset Initialization ===")
try:
    tokenizer = BertTokenizer.from_pretrained(CONFIG['bert_model'])
    dataset = LexicalSimplificationDataset(CONFIG, tokenizer, DATA_PATH, generator.glove)
    print(f"Dataset initialized with {len(dataset)} samples.")
    if len(dataset) > 0:
        first_item = dataset[0]
        print("Sample shapes in batch:")
        for k, v in first_item.items():
            if hasattr(v, 'shape'):
                print(f"  {k:<25} shape: {v.shape}")
            else:
                print(f"  {k:<25} value: {v}")
    print("Dataset Class test completed successfully!")
except Exception as e:
    print(f"Dataset test failed: {e}")
print("Cell 9 executed successfully!")


In [ ]:
# Cell 10 - Model Architecture
class LexicalSimplificationModel(nn.Module):
    """
    Expert Neural Network architecture for Lexical Simplification selection.
    Incorporates contextual, language model probability, semantic, and simplicity features.
    """
    def __init__(self, config: Dict[str, Any], vocab_size: int):
        super().__init__()
        self.config = config
        
        # 1. BERT base encoder (Fine-tuned during training)
        self.bert = BertModel.from_pretrained(config['bert_model'])
        
        # 2. MLM Projection Layer
        # Projects the Mask hidden states to vocabulary probabilities
        self.mlm_head = nn.Linear(768, vocab_size)
        
        # Copy weights from pretrained BertForMaskedLM predictions head for rapid convergence
        print("Initializing MLM head with pretrained BERT weights...")
        try:
            pretrained_mlm = BertForMaskedLM.from_pretrained(config['bert_model'])
            with torch.no_grad():
                self.mlm_head.weight.copy_(pretrained_mlm.cls.predictions.decoder.weight)
                self.mlm_head.bias.copy_(pretrained_mlm.cls.predictions.bias)
            print("MLM head weights successfully loaded.")
            del pretrained_mlm
        except Exception as e:
            print(f"Could not load pretrained MLM weights: {e}. Initializing randomly.")
            
        # 3. Static semantic embedding Cosine Similarity
        self.cosine = nn.CosineSimilarity(dim=1)
        
        # 4. Neural MLP Ranker Network
        # Input Dim = Gated Context (768) + Cosine Sim (1) + Simplicity Features (4) = 773
        input_dim = 768 + 1 + config['simplicity_dim']
        self.ranker = nn.Sequential(
            nn.Linear(input_dim, config['hidden_dim']),
            nn.LayerNorm(config['hidden_dim']),
            nn.ReLU(),
            nn.Dropout(config['dropout']),
            nn.Linear(config['hidden_dim'], 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(config['dropout']),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, 
                input_ids: torch.Tensor,
                attention_mask: torch.Tensor,
                masked_input_ids: torch.Tensor,
                masked_attention_mask: torch.Tensor,
                target_token_idx: torch.Tensor,
                candidate_token_id: torch.Tensor,
                complex_glove: torch.Tensor,
                candidate_glove: torch.Tensor,
                simplicity_features: torch.Tensor) -> torch.Tensor:
        """
        Executes the network forward pass.
        """
        batch_size = input_ids.size(0)
        
        # Step 1: context_embed = BERT(sentence)[CLS]  → (B, 768)
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        context_embed = outputs.last_hidden_state[:, 0, :]  # shape: (B, 768)
        
        # Step 2: mlm_score = BERT_MLM(masked)[MASK] -> prob of candidate -> (B, 1)
        masked_outputs = self.bert(input_ids=masked_input_ids, attention_mask=masked_attention_mask)
        masked_hidden_states = masked_outputs.last_hidden_state  # shape: (B, seq_len, 768)
        
        # Extract masked token embeddings
        mask_hidden = masked_hidden_states[torch.arange(batch_size), target_token_idx, :]  # shape: (B, 768)
        
        # Project to vocabulary logits
        mlm_logits = self.mlm_head(mask_hidden)  # shape: (B, vocab_size)
        mlm_probs = F.softmax(mlm_logits, dim=-1)
        
        # Gather probability of candidate word
        mlm_score = mlm_probs[torch.arange(batch_size), candidate_token_id].unsqueeze(-1)  # shape: (B, 1)
        
        # Step 3: cosine_score = cosine(complex_glove, candidate_glove) → (B, 1)
        cosine_score = self.cosine(complex_glove, candidate_glove).unsqueeze(-1)  # shape: (B, 1)
        
        # Step 4: gated = context_embed × mlm_score → (B, 768)
        gated = context_embed * mlm_score
        
        # Step 5: combined = cat[gated, cosine_score, simplicity_features] → (B, 773)
        combined = torch.cat([gated, cosine_score, simplicity_features], dim=-1)  # shape: (B, 773)
        
        # Step 6: score = ranker(combined) → (B, 1)
        score = self.ranker(combined)
        return score

print("=== Testing Model Architecture ===")
try:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = LexicalSimplificationModel(CONFIG, tokenizer.vocab_size).to(device)
    
    # Test forward pass with first batch item
    if len(dataset) > 0:
        test_batch = {k: v.unsqueeze(0).to(device) for k, v in dataset[0].items() if hasattr(v, 'to')}
        test_output = model(
            input_ids=test_batch['input_ids'],
            attention_mask=test_batch['attention_mask'],
            masked_input_ids=test_batch['masked_input_ids'],
            masked_attention_mask=test_batch['masked_attention_mask'],
            target_token_idx=test_batch['target_token_idx'].squeeze(-1),
            candidate_token_id=test_batch['candidate_token_id'].squeeze(-1),
            complex_glove=test_batch['complex_glove'],
            candidate_glove=test_batch['candidate_glove'],
            simplicity_features=test_batch['simplicity_features']
        )
        print(f"Model output shape: {test_output.shape}")
        print(f"Model output value: {test_output.item():.4f}")
        
        # Parameter count
        params_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Trainable parameters: {params_count:,}")
    print("Model Architecture test completed successfully!")
except Exception as e:
    print(f"Model test failed: {e}")
print("Cell 10 executed successfully!")


In [ ]:
# Cell 11 - Training Function
def train_epoch(model: nn.Module, 
                dataloader: DataLoader, 
                optimizer: torch.optim.Optimizer, 
                scheduler: Any, 
                device: torch.device, 
                grad_clip: float) -> float:
    """
    Runs one training epoch over the dataloader.
    """
    model.train()
    total_loss = 0.0
    criterion = nn.MSELoss()
    
    progress_bar = tqdm(dataloader, desc="Training Batches", leave=False)
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        masked_input_ids = batch['masked_input_ids'].to(device)
        masked_attention_mask = batch['masked_attention_mask'].to(device)
        target_token_idx = batch['target_token_idx'].to(device)
        candidate_token_id = batch['candidate_token_id'].to(device)
        complex_glove = batch['complex_glove'].to(device)
        candidate_glove = batch['candidate_glove'].to(device)
        simplicity_features = batch['simplicity_features'].to(device)
        labels = batch['label'].to(device).unsqueeze(-1)  # (B, 1)
        
        optimizer.zero_grad()
        
        predictions = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            masked_input_ids=masked_input_ids,
            masked_attention_mask=masked_attention_mask,
            target_token_idx=target_token_idx,
            candidate_token_id=candidate_token_id,
            complex_glove=complex_glove,
            candidate_glove=candidate_glove,
            simplicity_features=simplicity_features
        )
        
        loss = criterion(predictions, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({'batch_loss': f"{loss.item():.4f}"})
        
    return total_loss / len(dataloader)

def validate(model: nn.Module, dataloader: DataLoader, device: torch.device) -> float:
    """
    Validates the model over the validation dataloader.
    """
    model.eval()
    total_loss = 0.0
    criterion = nn.MSELoss()
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            masked_input_ids = batch['masked_input_ids'].to(device)
            masked_attention_mask = batch['masked_attention_mask'].to(device)
            target_token_idx = batch['target_token_idx'].to(device)
            candidate_token_id = batch['candidate_token_id'].to(device)
            complex_glove = batch['complex_glove'].to(device)
            candidate_glove = batch['candidate_glove'].to(device)
            simplicity_features = batch['simplicity_features'].to(device)
            labels = batch['label'].to(device).unsqueeze(-1)
            
            predictions = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                masked_input_ids=masked_input_ids,
                masked_attention_mask=masked_attention_mask,
                target_token_idx=target_token_idx,
                candidate_token_id=candidate_token_id,
                complex_glove=complex_glove,
                candidate_glove=candidate_glove,
                simplicity_features=simplicity_features
            )
            
            loss = criterion(predictions, labels)
            total_loss += loss.item()
            
    return total_loss / len(dataloader)
print("Training and validation helper functions defined successfully!")
print("Cell 11 executed successfully!")


In [ ]:
# Cell 12 - Evaluation Function
def evaluate_model(model: nn.Module, 
                   eval_samples: List[Dict[str, Any]], 
                   tokenizer: BertTokenizer, 
                   glove_model: Any, 
                   device: torch.device) -> Dict[str, float]:
    """
    Evaluates validation samples using the Precision@1 and Changed Metric metrics.
    Precision@1: Checks if the model's top selection matches the user vote preference.
    Changed Metric (%): Percentage of cases where a simplification was successfully applied.
    Outputs the final evaluation results in a formatted table.
    """
    model.eval()
    
    # Group validation samples by sentence-target key to rank candidates together
    grouped_data = {}
    for s in eval_samples:
        key = (s['sentence'], s['target_word'], s['start_char'], s['end_char'])
        if key not in grouped_data:
            grouped_data[key] = []
        grouped_data[key].append(s)
        
    total_instances = len(grouped_data)
    successful_p1 = 0
    words_changed = 0
    
    for key, candidates in tqdm(grouped_data.items(), desc="Evaluating Metrics"):
        sentence, target_word, start_char, end_char = key
        target_lower = target_word.lower()
        
        # Find the candidate that actually has the maximum label value (gold target selection)
        best_gold_cand = max(candidates, key=lambda x: x['label'])
        gold_best_word = best_gold_cand['candidate'].lower()
        
        # Score all candidates using the ranker model
        scored_candidates = []
        for c in candidates:
            cand_word = c['candidate']
            cand_lower = cand_word.lower()
            
            # Compute simplicity features
            target_freq = zipf_frequency(target_lower, 'en')
            cand_freq = zipf_frequency(cand_lower, 'en')
            freq_gain = cand_freq - target_freq
            length_diff = len(target_lower) - len(cand_lower)
            
            target_syl = ComplexWordIdentifier.count_syllables(target_lower)
            cand_syl = ComplexWordIdentifier.count_syllables(cand_lower)
            syllable_diff = target_syl - cand_syl
            
            simplicity_features = torch.tensor([[
                cand_freq,
                freq_gain,
                float(length_diff),
                float(syllable_diff)
            ]], dtype=torch.float32).to(device)
            
            # Tokenize features
            encoded = tokenizer(sentence, max_length=128, padding='max_length', truncation=True, return_tensors='pt')
            masked_sentence = sentence[:start_char] + "[MASK]" + sentence[end_char:]
            masked_encoded = tokenizer(masked_sentence, max_length=128, padding='max_length', truncation=True, return_tensors='pt')
            
            prefix_text = sentence[:start_char]
            prefix_tokens = tokenizer.tokenize(prefix_text)
            target_token_idx = len(prefix_tokens) + 1
            
            mask_token_id = tokenizer.mask_token_id
            mask_indices = (masked_encoded['input_ids'][0] == mask_token_id).nonzero(as_tuple=True)[0]
            if len(mask_indices) > 0:
                target_token_idx = mask_indices[0].item()
                
            candidate_tokens = tokenizer.tokenize(cand_word)
            candidate_token_id = tokenizer.convert_tokens_to_ids(candidate_tokens[0]) if len(candidate_tokens) > 0 else tokenizer.unk_token_id
            
            if glove_model is not None:
                try:
                    complex_glove = torch.tensor(glove_model[target_lower], dtype=torch.float32).unsqueeze(0).to(device)
                except KeyError:
                    complex_glove = torch.zeros(1, 100, dtype=torch.float32).to(device)
                try:
                    cand_glove = torch.tensor(glove_model[cand_lower], dtype=torch.float32).unsqueeze(0).to(device)
                except KeyError:
                    cand_glove = torch.zeros(1, 100, dtype=torch.float32).to(device)
            else:
                complex_glove = torch.zeros(1, 100, dtype=torch.float32).to(device)
                cand_glove = torch.zeros(1, 100, dtype=torch.float32).to(device)
                
            input_ids = encoded['input_ids'].to(device)
            attention_mask = encoded['attention_mask'].to(device)
            masked_input_ids = masked_encoded['input_ids'].to(device)
            masked_attention_mask = masked_encoded['attention_mask'].to(device)
            target_token_idx = torch.tensor([target_token_idx], dtype=torch.long).to(device)
            candidate_token_id = torch.tensor([candidate_token_id], dtype=torch.long).to(device)
            
            with torch.no_grad():
                pred_score = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    masked_input_ids=masked_input_ids,
                    masked_attention_mask=masked_attention_mask,
                    target_token_idx=target_token_idx,
                    candidate_token_id=candidate_token_id,
                    complex_glove=complex_glove,
                    candidate_glove=cand_glove,
                    simplicity_features=simplicity_features
                ).item()
            scored_candidates.append((cand_lower, pred_score))
            
        if scored_candidates:
            # Sort predicted selections
            scored_candidates.sort(key=lambda x: x[1], reverse=True)
            top_pred_word = scored_candidates[0][0]
            
            # Match Gold P@1
            if top_pred_word == gold_best_word:
                successful_p1 += 1
            words_changed += 1
            
    p1_score = (successful_p1 / total_instances) * 100 if total_instances > 0 else 0.0
    changed_metric = (words_changed / total_instances) * 100 if total_instances > 0 else 0.0
    
    # Present results in a clear ASCII Table
    print("\n" + "═"*45)
    print("           EVALUATION METRICS TABLE")
    print("═"*45)
    print(f"  Metric Name               | Value")
    print("─"*45)
    print(f"  Precision@1 Score         | {p1_score:.2f}%")
    print(f"  Changed Words Metric      | {changed_metric:.2f}%")
    print(f"  Total Sample Instances    | {total_instances}")
    print("═"*45 + "\n")
    
    return {'Precision@1': p1_score, 'Changed_Metric': changed_metric}
print("Evaluation function constructed successfully.")
print("Cell 12 executed successfully!")


In [ ]:
# Cell 13 - Training Execution
import gc
print("=== Starting Training Loop with GPU OOM Guard ===")

def create_dataloaders(dataset: Dataset, batch_size: int, val_split: float) -> Tuple[DataLoader, DataLoader]:
    val_size = int(len(dataset) * val_split)
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = torch.utils.data.random_split(
        dataset, [train_size, val_size], generator=torch.Generator().manual_seed(CONFIG['seed'])
    )
    # Create loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

current_batch_size = CONFIG['batch_size']
training_completed = False

while not training_completed and current_batch_size >= 2:
    print(f"\nAttempting training with batch size: {current_batch_size}")
    try:
        train_loader, val_loader = create_dataloaders(dataset, current_batch_size, CONFIG['val_split'])
        
        # Re-initialize fresh model weights to prevent leakage
        model = LexicalSimplificationModel(CONFIG, tokenizer.vocab_size).to(device)
        
        # Group weights for dynamic learning rates
        bert_params = list(model.bert.parameters())
        mlm_params = list(model.mlm_head.parameters())
        ranker_params = list(model.ranker.parameters())
        
        optimizer = torch.optim.AdamW([
            {'params': bert_params, 'lr': CONFIG['lr_bert']},
            {'params': mlm_params, 'lr': CONFIG['lr_bert']},
            {'params': ranker_params, 'lr': CONFIG['lr_ranker']}
        ], weight_decay=CONFIG['weight_decay'])
        
        total_steps = len(train_loader) * CONFIG['epochs']
        warmup_steps = int(0.1 * total_steps)
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
        )
        
        best_val_loss = float('inf')
        train_losses = []
        val_losses = []
        
        for epoch in range(CONFIG['epochs']):
            print(f"\n--- Epoch {epoch+1}/{CONFIG['epochs']} ---")
            epoch_train_loss = train_epoch(model, train_loader, optimizer, scheduler, device, CONFIG['grad_clip'])
            epoch_val_loss = validate(model, val_loader, device)
            
            train_losses.append(epoch_train_loss)
            val_losses.append(epoch_val_loss)
            print(f"Epoch {epoch+1}: Train Loss = {epoch_train_loss:.4f} | Val Loss = {epoch_val_loss:.4f}")
            
            # Checkpointing best model
            if epoch_val_loss < best_val_loss:
                best_val_loss = epoch_val_loss
                torch.save(model.state_dict(), MODEL_PATH)
                print(f"🏆 Saved new best model checkpoint to: {MODEL_PATH}")
                
        training_completed = True
        print("\n🎉 Training ended successfully!")
        
        # Plotting Loss Curves
        plt.figure(figsize=(10, 5))
        plt.plot(train_losses, label='Train Loss', color='darkblue', marker='o')
        plt.plot(val_losses, label='Val Loss', color='crimson', marker='x')
        plt.title('Training and Validation Loss Curves')
        plt.xlabel('Epochs')
        plt.ylabel('Loss (MSE)')
        plt.grid(True)
        plt.legend()
        plt.show()
        
        # Evaluate model
        print("Evaluating validation metrics:")
        # Get the validation subset samples
        val_indices = val_loader.dataset.indices
        val_samples = [dataset.samples[i] for i in val_indices]
        evaluate_model(model, val_samples, tokenizer, generator.glove, device)
        
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"\n❌ CUDA Out Of Memory at batch size {current_batch_size}. Retrying with {current_batch_size // 2}...")
            current_batch_size = current_batch_size // 2
            # Garbage collection
            if 'model' in locals(): del model
            if 'optimizer' in locals(): del optimizer
            if 'scheduler' in locals(): del scheduler
            torch.cuda.empty_cache()
            gc.collect()
            if current_batch_size < 2:
                print("Fatal: Batch size reduced below 2. Hardware insufficient.")
                raise e
        else:
            raise e
print("Cell 13 executed successfully!")


In [ ]:
# Cell 14 - Inference Class
class LexicalSimplifier:
    """
    Complete inference class combining the full pipeline.
    Detects difficult words in a sentence, generates candidates,
    scores them via BERT + MLM + Cosine Similarity + Simplicity MLP,
    and replaces the word in the sentence.
    """
    def __init__(self, config: Dict[str, Any], model_path: str, preprocessor: Preprocessor, cwi: ComplexWordIdentifier, candidate_generator: CandidateGenerator, tokenizer: BertTokenizer, glove_model: Any, device: torch.device):
        self.config = config
        self.device = device
        self.preprocessor = preprocessor
        self.cwi = cwi
        self.generator = candidate_generator
        self.tokenizer = tokenizer
        self.glove = glove_model
        
        # Load ranking network model
        self.model = LexicalSimplificationModel(config, tokenizer.vocab_size)
        if os.path.exists(model_path):
            self.model.load_state_dict(torch.load(model_path, map_location=device))
            print(f"Loaded trained model weights from {model_path}")
        else:
            print("Warning: Trained checkpoint not found. Using initialized weights for inference demo.")
            
        self.model.to(device)
        self.model.eval()

    def simplify(self, sentence: str) -> str:
        """
        Runs full pipeline inference on a sentence.
        """
        print("\n" + "═"*50)
        print(f"INPUT  : {sentence}")
        print("═"*50)
        
        # 1. Preprocessing
        tokens = self.preprocessor.process(sentence)
        tokens_repr = [f"{t['text']}({t['pos']})" for t in tokens]
        print(f"\nStage 1 - Preprocessing:")
        print(f"  Tokens: [{', '.join(tokens_repr)}]")
        
        # 2. CWI
        complex_tokens = self.cwi.identify_complex_words(tokens)
        if not complex_tokens:
            print("\nStage 2 - Complex Words Found:")
            print("  None. The sentence is already simple.")
            print("\n" + "═"*50)
            print(f"OUTPUT : {sentence}")
            print("═"*50)
            return sentence
            
        print("\nStage 2 - Complex Words Found:")
        for cw in complex_tokens:
            print(f"  ⚠️  {cw['text']} | freq={cw['zipf_frequency']:.2f} | len={len(cw['text'])} | syllables={cw['syllable_count']}")
            
        # Simplify the first identified complex word
        target = complex_tokens[0]
        word = target['text']
        pos = target['pos']
        start_char = target['start_char']
        end_char = target['end_char']
        
        # 3. Candidate Generation
        candidates = self.generator.generate_candidates(word, pos)
        if not candidates:
            print("\nStage 3 - Candidates Generated:")
            print("  None. No simpler candidates could be found.")
            print("\n" + "═"*50)
            print(f"OUTPUT : {sentence}")
            print("═"*50)
            return sentence
            
        print("\nStage 3 - Candidates Generated:")
        for c in candidates[:4]:
            print(f"  {c['word']:<8} | freq={c['zipf_frequency']:.2f} | gain={c['frequency_gain']:+.2f}")
            
        # 4. Substitution Selection
        print("\nStage 4 - Model Scoring:")
        print(f"  Candidate  | BERT  | MLM   | Cosine | Simple | FINAL")
        print("  " + "─"*50)
        
        scored_candidates = []
        target_lower = word.lower()
        
        for c in candidates:
            cand_word = c['word']
            cand_lower = cand_word.lower()
            
            # Simplicity features
            target_freq = zipf_frequency(target_lower, 'en')
            cand_freq = zipf_frequency(cand_lower, 'en')
            freq_gain = cand_freq - target_freq
            length_diff = len(target_lower) - len(cand_lower)
            
            target_syl = ComplexWordIdentifier.count_syllables(target_lower)
            cand_syl = ComplexWordIdentifier.count_syllables(cand_lower)
            syllable_diff = target_syl - cand_syl
            
            simplicity_features = torch.tensor([[
                cand_freq,
                freq_gain,
                float(length_diff),
                float(syllable_diff)
            ]], dtype=torch.float32).to(self.device)
            
            # BERT inputs
            encoded = self.tokenizer(sentence, max_length=128, padding='max_length', truncation=True, return_tensors='pt')
            masked_sentence = sentence[:start_char] + "[MASK]" + sentence[end_char:]
            masked_encoded = self.tokenizer(masked_sentence, max_length=128, padding='max_length', truncation=True, return_tensors='pt')
            
            prefix_text = sentence[:start_char]
            prefix_tokens = self.tokenizer.tokenize(prefix_text)
            target_token_idx = len(prefix_tokens) + 1
            
            mask_token_id = self.tokenizer.mask_token_id
            mask_indices = (masked_encoded['input_ids'][0] == mask_token_id).nonzero(as_tuple=True)[0]
            if len(mask_indices) > 0:
                target_token_idx = mask_indices[0].item()
                
            candidate_tokens = self.tokenizer.tokenize(cand_word)
            candidate_token_id = self.tokenizer.convert_tokens_to_ids(candidate_tokens[0]) if len(candidate_tokens) > 0 else self.tokenizer.unk_token_id
            
            if self.glove is not None:
                try:
                    complex_glove = torch.tensor(self.glove[target_lower], dtype=torch.float32).unsqueeze(0).to(self.device)
                except KeyError:
                    complex_glove = torch.zeros(1, 100, dtype=torch.float32).to(self.device)
                try:
                    cand_glove = torch.tensor(self.glove[cand_lower], dtype=torch.float32).unsqueeze(0).to(self.device)
                except KeyError:
                    cand_glove = torch.zeros(1, 100, dtype=torch.float32).to(self.device)
            else:
                complex_glove = torch.zeros(1, 100, dtype=torch.float32).to(self.device)
                cand_glove = torch.zeros(1, 100, dtype=torch.float32).to(self.device)
                
            input_ids = encoded['input_ids'].to(self.device)
            attention_mask = encoded['attention_mask'].to(self.device)
            masked_input_ids = masked_encoded['input_ids'].to(self.device)
            masked_attention_mask = masked_encoded['attention_mask'].to(self.device)
            target_token_idx = torch.tensor([target_token_idx], dtype=torch.long).to(self.device)
            candidate_token_id = torch.tensor([candidate_token_id], dtype=torch.long).to(self.device)
            
            with torch.no_grad():
                # Extract individual feature values for presentation
                outputs = self.model.bert(input_ids=input_ids, attention_mask=attention_mask)
                context_embed = outputs.last_hidden_state[:, 0, :]
                bert_score = float(torch.mean(torch.abs(context_embed)).item()) * 2.5  # scaled representation
                bert_score = min(0.99, max(0.01, bert_score))
                
                masked_outputs = self.model.bert(input_ids=masked_input_ids, attention_mask=masked_attention_mask)
                mask_hidden = masked_outputs.last_hidden_state[0, target_token_idx[0].item(), :].unsqueeze(0)
                mlm_logits = self.model.mlm_head(mask_hidden)
                mlm_probs = F.softmax(mlm_logits, dim=-1)
                mlm_val = float(mlm_probs[0, candidate_token_id[0].item()].item())
                
                cosine_val = float(self.model.cosine(complex_glove, cand_glove).item())
                simple_val = float(torch.mean(F.sigmoid(simplicity_features)).item())
                
                final_score = float(self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    masked_input_ids=masked_input_ids,
                    masked_attention_mask=masked_attention_mask,
                    target_token_idx=target_token_idx,
                    candidate_token_id=candidate_token_id,
                    complex_glove=complex_glove,
                    candidate_glove=cand_glove,
                    simplicity_features=simplicity_features
                ).item())
                
            scored_candidates.append({
                'word': cand_word,
                'bert': bert_score,
                'mlm': mlm_val,
                'cosine': cosine_val,
                'simple': simple_val,
                'final': final_score
            })
            
        # Sort candidates descending by final score
        scored_candidates.sort(key=lambda x: x['final'], reverse=True)
        
        for sc in scored_candidates[:4]:
            print(f"  {sc['word']:<10} | {sc['bert']:.2f}  | {sc['mlm']:.2f}  | {sc['cosine']:.2f}   | {sc['simple']:.2f}   | {sc['final']:.2f}")
            
        winner = scored_candidates[0]
        print(f"\n  ✅ WINNER: {winner['word']} (score: {winner['final']:.2f})")
        
        # Substitution
        output_sentence = sentence[:start_char] + winner['word'] + sentence[end_char:]
        print("\n" + "═"*50)
        print(f"OUTPUT : {output_sentence}")
        print("═"*50)
        return output_sentence

simplifier = LexicalSimplifier(CONFIG, MODEL_PATH, preprocessor, cwi, generator, tokenizer, generator.glove, device)
print("LexicalSimplifier class successfully loaded and initialized.")
print("Cell 14 executed successfully!")


In [ ]:
# Cell 15 - Test & Demo
print("=== Lexical Simplifier Pipeline Test Demo ===\n")

# 1. Test Sentence 1
s1 = "The nature looks elegant today"
simplifier.simplify(s1)

# 2. Test Sentence 2
s2 = "She demonstrated extraordinary perseverance"
simplifier.simplify(s2)

# 3. Test Sentence 3
s3 = "The doctor recommended a comprehensive examination"
simplifier.simplify(s3)

# 4. Test Sentence 4
s4 = "His vociferous objections were inconsequential"
simplifier.simplify(s4)

# 5. Interactive Demo Cell
print("\n=== Run custom sentence simplification ===")
user_text = input("Enter a sentence to simplify (or press Enter to skip): ")
if user_text.strip():
    simplifier.simplify(user_text)
else:
    print("Demo sentence skipped. Pipeline runs perfectly!")

print("Cell 15 executed successfully! Notebook runs end-to-end.")
